# 5-2 문제 유형별 손실 함수 선택

강의 원문 대신 직접 작성한 코드, 실행 결과와 학습 메모를 정리했습니다.


In [1]:
import random
import json
import math
import shutil
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, Dataset, random_split

# 실습 결과가 매번 비슷하게 나오도록 seed를 고정합니다.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

device: cpu


In [3]:
def get_loss_fn(task_type):
    # TODO: task_type 값에 맞춰 loss 함수를 반환하세요.
    if task_type == 'regression':
        return nn.MSELoss()  # 임시 코드입니다.
    if task_type == 'binary':
        return nn.BCEWithLogitsLoss()  # 임시 코드입니다.
    if task_type == 'multiclass':
        return nn.CrossEntropyLoss()  # 임시 코드입니다.
    raise ValueError('unknown task_type')

for task in ['regression', 'binary', 'multiclass']:
    print(task, '->', get_loss_fn(task).__class__.__name__)

regression -> MSELoss
binary -> BCEWithLogitsLoss
multiclass -> CrossEntropyLoss


In [7]:
binary_logits = torch.randn(4, 1)
binary_target_raw = torch.tensor([1, 0, 1, 0])
multiclass_logits = torch.randn(4, 3)
multiclass_target_raw = torch.tensor([[0], [2], [1], [0]])

# TODO: BCEWithLogitsLoss에 맞게 target을 수정하세요.
binary_target = binary_target_raw.float().unsqueeze(1)

# TODO: CrossEntropyLoss에 맞게 target을 수정하세요.
multiclass_target = multiclass_target_raw.long().squeeze(1)

try:
    bce = nn.BCEWithLogitsLoss()(binary_logits, binary_target)
    ce = nn.CrossEntropyLoss()(multiclass_logits, multiclass_target)
    print('bce:', float(bce), 'ce:', float(ce))
except Exception as e:
    print('수정이 필요합니다:', type(e).__name__, e)

bce: 0.6279362440109253 ce: 1.3022699356079102


In [15]:
###

def compute_loss(task_type, logits, target):
    if task_type == 'regression':
        target = target.float().view_as(logits)
        return nn.MSELoss()(logits, target)
    if task_type == 'binary':
        target = target.float().view_as(logits)
        return nn.BCEWithLogitsLoss()(logits, target)
    if task_type == 'multiclass':
        target = target.view(-1).long()
        return nn.CrossEntropyLoss()(logits, target)
    raise ValueError(task_type)

print('regression:', compute_loss('regression', torch.randn(3,1), torch.randn(3,1)))
print('binary:', compute_loss('binary', torch.randn(3,1), torch.tensor([1,0,1])))
print('multiclass:', compute_loss('multiclass', torch.randn(3,4), torch.tensor([[1],[2],[0]])))

regression: tensor(1.7813)
binary: tensor(0.4975)
multiclass: tensor(1.0964)
